### Model Comparison and Selection 

##### Google Colab to mount Google Drive to our Colab environment.

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


From google.colab import drive: imports the drive module from the google.colab package. Google Colab is a cloud-based Jupyter notebook environment, and this module allows us to interact with your Google Drive from Colab.

drive.mount('/content/drive'): This command mounts our Google Drive to the Colab environment under the path /content/drive. 

In [29]:
import pandas as pd

with open('/content/drive/My Drive/final_tokens_labels.csv', 'r') as file:
    lines = file.readlines()
data = [line.strip().split('\t') for line in lines]
data = pd.DataFrame(data)

In [30]:
print("First few rows of the dataset:")
print(data.head(15))
print("Columns in the dataset:", data.columns)


First few rows of the dataset:
                  0
0       Token,Label
1   የህፃናት,B-Product
2   መመገቢያ,I-Product
3      ጡጦ,I-Product
4        ዋጋ,B-PRICE
5       400,I-PRICE
6        ብር,I-PRICE
7             ውስን,O
8              ፍሬ,O
9              ነው,O
10            ያለው,O
11           አድራሻ,O
12     ቁ1መገናኛ,B-LOC
13       መሰረት,I-LOC
14        ደፋር,I-LOC
Columns in the dataset: RangeIndex(start=0, stop=1, step=1)


#### Data Preparation 

In [31]:
data = pd.read_csv('/content/drive/My Drive/final_tokens_labels.csv', header=None)
data = data.drop(index=0)
data.columns = ['Token','Label']
print("Cleaned Dataset:")
print(data.head(10))

Cleaned Dataset:
    Token      Label
1   የህፃናት  B-Product
2   መመገቢያ  I-Product
3      ጡጦ  I-Product
4      ዋጋ    B-PRICE
5     400    I-PRICE
6      ብር    I-PRICE
7     ውስን          O
8      ፍሬ          O
9      ነው          O
10    ያለው          O


### Tokenize 

Tokenize text data (tokens and labels) for three different models—XLM-Roberta, DistilBERT, and mBERT—for a Named Entity Recognition (NER) task. Here’s a breakdown of what each part of the code does:

In [32]:
import pandas as pd
from transformers import XLMRobertaTokenizerFast, DistilBertTokenizerFast, BertTokenizerFast
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
file_path = '/content/drive/My Drive/final_tokens_labels.csv'
data = pd.read_csv(file_path, header=None)
data = data.drop(index=0)
data.columns = ['Token', 'Label']
data['Label'] = data['Label'].fillna('O')
xlm_roberta_tokenizer = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-multilingual-cased')
mbert_tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')
label_mapping = {'B-Product': 0, 'I-Product': 1, 'B-PRICE': 2, 'I-PRICE': 3,
                 'B-LOC': 4, 'I-LOC': 5, 'O': 6}
def tokenize_and_align_labels(tokenizer, tokens, labels, label_mapping):
    tokenized_inputs = tokenizer(
        tokens.tolist(),
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )

    aligned_labels = []
    for i in range(len(tokenized_inputs['input_ids'])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Get the mapping of tokens to original words

        label_ids = [-100 if word_id is None else label_mapping[labels.iloc[word_id]] for word_id in word_ids]
        aligned_labels.append(label_ids)

    tokenized_inputs['labels'] = aligned_labels
    return tokenized_inputs
tokens = data['Token']
labels = data['Label']

xlm_roberta_inputs = tokenize_and_align_labels(xlm_roberta_tokenizer, tokens, labels, label_mapping)
distilbert_inputs = tokenize_and_align_labels(distilbert_tokenizer, tokens, labels, label_mapping)
mbert_inputs = tokenize_and_align_labels(mbert_tokenizer, tokens, labels, label_mapping)
print("XLM-Roberta Inputs:", xlm_roberta_inputs)
print("DistilBERT Inputs:", distilbert_inputs)
print("mBERT Inputs:", mbert_inputs)


XLM-Roberta Inputs: {'input_ids': tensor([[     0, 114866,  29307,  32465,      6, 149322,   5430,  33609,      6,
          23380,  27339,  80667,   4082,  35648,      6, 100579,    548,  29831,
          13199,   3053,  23683, 140042,  47885,    418,   6231,  60014,   9171,
          88827,  21368, 191973,  45694,   2202,      6, 115742,      6,  31531,
           5653, 137526,  47885,   4777,   9016,  47885,    304,  60204,   4059,
           4047,      6, 186668, 221847,      6,   3851,  71429,    454,   3348,
          13851,   2010,    454,  47825,   5040,    454,  49334,  29307,    106,
           9171,      6,  31531,   5653,      6,  14414,   5653,  47885,    106,
          36225,   3894,   9550,  89709, 191539,   3894,   3882, 158332, 161025,
              6,  60204,   4059,   4047, 145960,    548,  15741,   4722,   3894,
           8894, 129176, 105619,    728,  13253,   9039,   7872,   2934, 169422,
          91602,  16333, 124449, 243084,    623,  17680,  10824,  70317,   

In [33]:
!pip install datasets


Prepare the tokenized inputs (from XLM-Roberta, DistilBERT, and mBERT) for training by converting them into a format suitable for model fine-tuning. 

In [34]:
from datasets import Dataset
def create_dataset(tokenized_inputs):
    return Dataset.from_dict({
        'input_ids': tokenized_inputs['input_ids'],
        'attention_mask': tokenized_inputs['attention_mask'],
        'labels': tokenized_inputs['labels'],
    })
xlm_roberta_dataset = create_dataset(xlm_roberta_inputs)
distilbert_dataset = create_dataset(distilbert_inputs)
mbert_dataset = create_dataset(mbert_inputs)
print("XLM-Roberta Dataset Example:", xlm_roberta_dataset[0])

XLM-Roberta Dataset Example: {'input_ids': [0, 114866, 29307, 32465, 6, 149322, 5430, 33609, 6, 23380, 27339, 80667, 4082, 35648, 6, 100579, 548, 29831, 13199, 3053, 23683, 140042, 47885, 418, 6231, 60014, 9171, 88827, 21368, 191973, 45694, 2202, 6, 115742, 6, 31531, 5653, 137526, 47885, 4777, 9016, 47885, 304, 60204, 4059, 4047, 6, 186668, 221847, 6, 3851, 71429, 454, 3348, 13851, 2010, 454, 47825, 5040, 454, 49334, 29307, 106, 9171, 6, 31531, 5653, 6, 14414, 5653, 47885, 106, 36225, 3894, 9550, 89709, 191539, 3894, 3882, 158332, 161025, 6, 60204, 4059, 4047, 145960, 548, 15741, 4722, 3894, 8894, 129176, 105619, 728, 13253, 9039, 7872, 2934, 169422, 91602, 16333, 124449, 243084, 623, 17680, 10824, 70317, 816, 24163, 29307, 23374, 2237, 194285, 162086, 68980, 32763, 104135, 62284, 3348, 131221, 81620, 64921, 2237, 45203, 12611, 4974, 31867, 4656, 5617, 227176, 2350, 72721, 4047, 41659, 6759, 82956, 548, 148227, 5617, 210139, 220554, 231990, 106494, 46945, 2302, 11484, 4413, 4047, 19850

In [35]:
!pip install transformers


#### Define Training Arguments

In [36]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)


#### Fine-tuning the XLM-Roberta model

In [38]:
from transformers import Trainer, XLMRobertaForTokenClassification
xlm_roberta_model = XLMRobertaForTokenClassification.from_pretrained('xlm-roberta-base', num_labels=len(label_mapping))
xlm_roberta_trainer = Trainer(
    model=xlm_roberta_model,
    args=training_args,
    train_dataset=xlm_roberta_dataset,
    eval_dataset=xlm_roberta_dataset
)

xlm_roberta_trainer.train()
xlm_roberta_metrics = xlm_roberta_trainer.evaluate()
print("XLM-Roberta Evaluation Metrics:", xlm_roberta_metrics)


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,1.853496
2,No log,1.797611
3,No log,1.769499


XLM-Roberta Evaluation Metrics: {'eval_loss': 1.76949942111969, 'eval_runtime': 1.8702, 'eval_samples_per_second': 0.535, 'eval_steps_per_second': 0.535, 'epoch': 3.0}


The decreasing validation loss over the epochs suggests that the model is improving its ability to generalize to unseen data.

#### Fine-Tuning the DistilBert

In [39]:
from transformers import DistilBertForTokenClassification

# Load the model
distilbert_model = DistilBertForTokenClassification.from_pretrained('distilbert-base-multilingual-cased', num_labels=len(label_mapping))

# Create the Trainer instance
distilbert_trainer = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=distilbert_dataset,
    eval_dataset=distilbert_dataset  # Ideally, use a separate validation set
)

# Train the model
distilbert_trainer.train()

# Evaluate the model
distilbert_metrics = distilbert_trainer.evaluate()
print("DistilBERT Evaluation Metrics:", distilbert_metrics)


model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,1.477893
2,No log,1.362034
3,No log,1.310360


DistilBERT Evaluation Metrics: {'eval_loss': 1.3103598356246948, 'eval_runtime': 0.9456, 'eval_samples_per_second': 1.058, 'eval_steps_per_second': 1.058, 'epoch': 3.0}


#### Fine-Tuning the mbert 

In [40]:
from transformers import BertForTokenClassification
mbert_model = BertForTokenClassification.from_pretrained('bert-base-multilingual-cased', num_labels=len(label_mapping))
mbert_trainer = Trainer(
    model=mbert_model,
    args=training_args,
    train_dataset=mbert_dataset,
    eval_dataset=mbert_dataset  # Ideally, use a separate validation set
)
mbert_trainer.train()
mbert_metrics = mbert_trainer.evaluate()
print("mBERT Evaluation Metrics:", mbert_metrics)


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,1.563445
2,No log,1.366571
3,No log,1.279550


mBERT Evaluation Metrics: {'eval_loss': 1.2795498371124268, 'eval_runtime': 2.426, 'eval_samples_per_second': 0.412, 'eval_steps_per_second': 0.412, 'epoch': 3.0}


In [41]:
print("XLM-Roberta Evaluation Metrics:")
print(xlm_roberta_metrics)
print("DistilBERT Evaluation Metrics:")
print(distilbert_metrics)
print("mBERT Evaluation Metrics:")
print(mbert_metrics)


XLM-Roberta Evaluation Metrics:
{'eval_loss': 1.76949942111969, 'eval_runtime': 1.8702, 'eval_samples_per_second': 0.535, 'eval_steps_per_second': 0.535, 'epoch': 3.0}
DistilBERT Evaluation Metrics:
{'eval_loss': 1.3103598356246948, 'eval_runtime': 0.9456, 'eval_samples_per_second': 1.058, 'eval_steps_per_second': 1.058, 'epoch': 3.0}
mBERT Evaluation Metrics:
{'eval_loss': 1.2795498371124268, 'eval_runtime': 2.426, 'eval_samples_per_second': 0.412, 'eval_steps_per_second': 0.412, 'epoch': 3.0}


XLM-Roberta achieved an evaluation loss of 1.7695, which is significantly higher than the losses of DistilBERT (1.3104) and mBERT (1.2795), suggesting that both DistilBERT and mBERT are better at minimizing loss and thus likely more effective in accurately identifying entities. In terms of efficiency, DistilBERT outperformed the other models in evaluation runtime, processing approximately 1.058 samples per second, compared to XLM-Roberta's 0.535 and mBERT's 0.412 samples per second. This indicates that DistilBERT is not only faster but also more efficient, making it a suitable choice for applications requiring real-time processing. Overall, based on the evaluation metrics, DistilBERT emerges as the best-performing model, combining lower loss with higher processing speed, which could be particularly beneficial for deployment in practical scenarios.